# 3. Policy Iteration
### Alternates Policy Evaluation and Improvement

**Policy Iteration** starts with an arbitrary policy and repeatedly alternates between:
1. **Policy Evaluation**: Fully evaluating $V^\pi(s)$ until convergence.
2. **Policy Improvement**: Generating $\pi' = \text{greedy}(V^\pi)$.

This cycle repeats until the policy stops changing (i.e. policy becomes stable), guaranteeing convergence to the **optimal policy $\pi^*$** and **optimal value function $V^*$**.

$$\pi_0 \xrightarrow{\text{eval}} V^{\pi_0} \xrightarrow{\text{improve}} \pi_1 \xrightarrow{\text{eval}} V^{\pi_1} \xrightarrow{\text{improve}} \dots \xrightarrow{} \pi^* \xrightarrow{\text{eval}} V^*$$


In [ ]:
# Step 1: Import Libraries
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
# Step 2: Define a 4x4 GridWorld MDP Environment
# States: 16 grid cells (0 to 15), where (0,0) [state 0] and (3,3) [state 15] are terminal
# Actions: 0: UP, 1: RIGHT, 2: DOWN, 3: LEFT

class GridWorld:
    def __init__(self, size=4):
        self.size = size
        self.num_states = size * size
        self.num_actions = 4
        self.actions = ['UP', 'RIGHT', 'DOWN', 'LEFT']
        self.terminal_states = {0, self.num_states - 1}
        
    def get_next_state_and_reward(self, state, action):
        if state in self.terminal_states:
            return state, 0.0
            
        r, c = divmod(state, self.size)
        if action == 0:    # UP
            r = max(r - 1, 0)
        elif action == 1:  # RIGHT
            c = min(c + 1, self.size - 1)
        elif action == 2:  # DOWN
            r = min(r + 1, self.size - 1)
        elif action == 3:  # LEFT
            c = max(c - 1, 0)
            
        next_state = r * self.size + c
        reward = -1.0  # -1 step cost until terminal goal reached
        return next_state, reward

env = GridWorld()
print(f"GridWorld initialized with {env.num_states} states and {env.num_actions} actions.")


In [ ]:
# Step 3: Complete Policy Iteration Algorithm
def policy_iteration(env, gamma=1.0, theta=1e-4):
    # Initialize random deterministic policy
    policy = np.ones((env.num_states, env.num_actions)) / env.num_actions
    V = np.zeros(env.num_states)
    
    iteration_count = 0
    
    while True:
        iteration_count += 1
        print(f"--- Policy Iteration Loop #{iteration_count} ---")
        
        # 1. Policy Evaluation
        eval_steps = 0
        while True:
            delta = 0.0
            V_new = np.copy(V)
            for s in range(env.num_states):
                if s in env.terminal_states:
                    continue
                v_s = sum(
                    policy[s][a] * (env.get_next_state_and_reward(s, a)[1] + gamma * V[env.get_next_state_and_reward(s, a)[0]])
                    for a in range(env.num_actions)
                )
                V_new[s] = v_s
                delta = max(delta, abs(V[s] - V_new[s]))
            V = V_new
            eval_steps += 1
            if delta < theta:
                break
        print(f"  Policy Evaluation converged in {eval_steps} sweeps.")
        
        # 2. Policy Improvement
        policy_stable = True
        for s in range(env.num_states):
            if s in env.terminal_states:
                continue
                
            old_action = np.argmax(policy[s])
            
            # Compute action values
            action_values = np.zeros(env.num_actions)
            for a in range(env.num_actions):
                next_s, r = env.get_next_state_and_reward(s, a)
                action_values[a] = r + gamma * V[next_s]
                
            best_action = np.argmax(action_values)
            
            # Check stability
            if old_action != best_action:
                policy_stable = False
                
            # Update policy to be greedy
            policy[s] = np.zeros(env.num_actions)
            policy[s][best_action] = 1.0
            
        if policy_stable:
            print(f"Optimal Policy reached and stabilized after {iteration_count} iterations!")
            break
            
    return policy, V

optimal_policy, optimal_V = policy_iteration(env)


In [ ]:
# Function to plot State-Value Heatmap and Policy Arrows
def plot_grid(V, policy=None, title="State-Value Grid"):
    fig, ax = plt.subplots(figsize=(6, 6))
    grid_v = V.reshape((4, 4))
    
    cax = ax.matshow(grid_v, cmap='Blues_r')
    fig.colorbar(cax)
    
    arrows = {0: '↑', 1: '→', 2: '↓', 3: '←'}
    
    for r in range(4):
        for c in range(4):
            s = r * 4 + c
            val_text = f"{V[s]:.1f}"
            if s in [0, 15]:
                val_text += "\n(Goal)"
            elif policy is not None:
                best_a = np.argmax(policy[s])
                val_text += f"\n{arrows[best_a]}"
                
            ax.text(c, r, val_text, ha='center', va='center', color='black', fontsize=11, weight='bold')
            
    ax.set_xticks(range(4))
    ax.set_yticks(range(4))
    plt.title(title, pad=20)
    plt.show()


In [ ]:
# Step 5: Visualize Optimal Policy and Value Function
plot_grid(optimal_V, policy=optimal_policy, title="Policy Iteration: Optimal Policy π* & V*")


### Summary
- Usually converges in very few outer policy iteration steps (often < 10).
- Each step requires a full iterative policy evaluation inner loop until convergence.
